# 08 · More one-hot copies, then retune CatBoost

**Starting point** (`07`): tuned CatBoost + 4 categorical copies (`C_NUM_DRIVERS`, `C_VEH_N_ROWS`, `C_CAT_ZONE`, `C_QUOTE_YEAR`), one-hot encoded. 15-fold CV AUC **0.7143**. The gain came from per-level one-hot splits on small-integer columns.

**Part 1, extend the trick** (seed-0 folds, paired Δ vs. the `07` model): one-hot copies of other small-integer columns (`features.CATS2`), coarse bands of credit / mean driver age / price, and dropping the numeric originals of the 4 copies. `one_hot_max_size = 12` so quote month (12 levels) is one-hot too; nothing in the base set has 11–12 levels, so this equals the `07` setting for the base. Passing groups are combined and confirmed on 15 folds (seeds 1–2 decide).

**Part 2, retune** CatBoost on the resulting feature set: 40 Optuna trials on seed-0 folds (lr 0.05), confirmed on 15 folds at lr 0.03. Adopt only with > 2 se on seeds 1–2. Then a 5-seed holdout submission.

In [1]:
import json, sys
import numpy as np, pandas as pd, optuna
sys.path.insert(0, '.')
import common as C, features as FE
optuna.logging.set_verbosity(optuna.logging.WARNING)

lab, hold = C.load()
L = FE.add_cats2(FE.add_static(lab))
PC = json.load(open('params_cats.json'))
F0, P0 = PC['features'], {k: v for k, v in PC['params'].items()}       # 07 model: 41 + 4 copies, ohm 10
P12 = P0 | {'one_hot_max_size': 12}
COPIES = [f'C_{c}' for c in FE.CAT_COPIES]
base = C.cv_score('cats_copies_ohm10', L, F0, 'cb', P0, seeds=(0,), log=False)          # cached from 07
base15 = C.cv_score('cats15___copies__ohm_10', L, F0, 'cb', P0, notes='07_catboost_cats')  # cached from 07
fmt = {'auc': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile': '{:.1%}'}

def paired(r, b):
    d = r['fold_aucs'] - b['fold_aucs']
    return {'auc': r['auc_mean'], 'Δ auc': d.mean(), 'se Δ': d.std() / np.sqrt(len(d)), 'top_decile': r['top_decile_prec'], 'rounds': r['rounds']}

def run(name, feats, params=P12):
    return paired(C.cv_score(f'cats2_{name}', L, feats, 'cb', params, seeds=(0,), log=False), base)
print(f'07 model: {len(F0)} features, seed-0 AUC {base["auc_mean"]:.4f}, 15-fold {base15["auc_mean"]:.4f}')

07 model: 45 features, seed-0 AUC 0.7145, 15-fold 0.7143


## Part 1 · More one-hot copies (seed-0 folds)

In [2]:
V = {g: run(g.replace(' ', '_').replace('/', '_'), F0 + cols) | {'n': len(cols)} for g, cols in FE.CATS2.items()}
NUMS = list(FE.CAT_COPIES)
V['drop numeric originals of the 4 copies'] = run('drop_orig', [f for f in F0 if f not in NUMS]) | {'n': -len(NUMS)}
tbl = pd.DataFrame(V).T
tbl.style.format(fmt)

,auc,Δ auc,se Δ,top_decile,rounds,n
young drivers,0.7136,-0.0009,0.0005,30.3%,402.000000,1.000000
policy veh counts,0.7147,+0.0002,0.0003,29.9%,524.000000,4.000000
prior claims,0.7132,-0.0014,0.0002,29.7%,400.000000,1.000000
quote month,0.7135,-0.0010,0.0004,29.9%,433.000000,1.000000
drivers - vehicles,0.7135,-0.0011,0.0004,29.9%,498.000000,1.000000
driver status counts,0.7133,-0.0012,0.0004,30.3%,432.000000,4.000000
binned credit/age/price,0.7137,-0.0008,0.0009,29.9%,386.000000,3.000000
drop numeric originals of the 4 copies,0.7111,-0.0034,0.0011,29.3%,486.000000,-4.000000


In [3]:
# Combine every group with Δ > se, confirm the combination (and the best single group) on 15 folds.
passing = [g for g in FE.CATS2 if tbl.loc[g, 'Δ auc'] > tbl.loc[g, 'se Δ']]
drop_orig = tbl.loc['drop numeric originals of the 4 copies', 'Δ auc'] > tbl.loc['drop numeric originals of the 4 copies', 'se Δ']
def feats_for(groups, drop):
    f = F0 + [c for g in groups for c in FE.CATS2[g]]
    return [x for x in f if not (drop and x in NUMS)]
cands = {}
if passing or drop_orig:
    cands['all passing'] = feats_for(passing, drop_orig)
    best_single = tbl.drop('drop numeric originals of the 4 copies').sort_values('Δ auc', ascending=False).index[0]
    if passing and [best_single] != passing:
        cands[f'best single ({best_single})'] = feats_for([best_single], False)
print('passing groups:', passing, '| drop originals:', drop_orig)
conf = {}
for i, (name, feats) in enumerate(cands.items()):
    r = C.cv_score(f'cats2_15_{i}_{len(feats)}', L, feats, 'cb', P12, notes='08 part 1')
    conf[name] = paired(r, base15) | {'Δ seeds 1-2': (r['fold_aucs'] - base15['fold_aucs'])[5:].mean(),
                                      'se seeds 1-2': (r['fold_aucs'] - base15['fold_aucs'])[5:].std() / np.sqrt(10), 'n_feats': len(feats)}
conf = pd.DataFrame(conf).T
display(conf.style.format(fmt | {'Δ seeds 1-2': '{:+.4f}', 'se seeds 1-2': '{:.4f}'}))
win = conf[conf['Δ seeds 1-2'] > 2 * conf['se seeds 1-2']].sort_values('Δ seeds 1-2', ascending=False) if len(conf) else conf
F1, P1 = (cands[win.index[0]], P12) if len(win) else (F0, P0)
print('Part 1 result:', win.index[0] if len(win) else 'no change (07 feature set kept)', '|', len(F1), 'features')

passing groups: [] | drop originals: False


Part 1 result: no change (07 feature set kept) | 45 features


**Reading:** no further copy helps. Only the policy-table vehicle counts are neutral (+0.0002 ± 0.0003); young drivers, prior claims, month, drivers − vehicles, driver status counts and coarse credit/age/price bands all lower AUC (−0.0008 to −0.0014). **Dropping the numeric originals hurts** (−0.0034): the model needs both forms, numeric thresholds *and* per-level splits. The gain in `07` came from these four specific columns, where conversion steps by level.

## Part 2 · Retune CatBoost on the Part-1 feature set
Same search space as `04_tuning` (depth, L2, random strength, bagging temperature, border count), plus `one_hot_max_size` ∈ {12, 32} to allow one-hot on the larger new columns. 40 TPE trials on seed-0 folds at lr 0.05; the best is re-run on 15 folds at lr 0.03. The comparison model is the Part-1 result with the old tuned parameters.

In [4]:
def cb_space(t):
    return dict(learning_rate=0.05, depth=t.suggest_int('depth', 3, 8), l2_leaf_reg=t.suggest_float('l2_leaf_reg', 1, 30, log=True),
                random_strength=t.suggest_float('random_strength', 0.1, 10, log=True),
                bagging_temperature=t.suggest_float('bagging_temperature', 0, 3),
                border_count=t.suggest_categorical('border_count', [32, 64, 128, 254]),
                one_hot_max_size=t.suggest_categorical('one_hot_max_size', [12, 32]))

def objective(t):
    r = C.cv_score('trial', L, F1, 'cb', P1 | cb_space(t), seeds=(0,), log=False, cache=False)
    t.set_user_attr('rounds', r['rounds'])
    return r['auc_mean']

st = optuna.create_study(study_name=f'cb_cats2_{len(F1)}', storage='sqlite:///optuna.db', direction='maximize', load_if_exists=True,
                         sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10))
st.enqueue_trial({k: P1.get(k, 12) for k in ['depth', 'l2_leaf_reg', 'random_strength', 'bagging_temperature', 'border_count']}
                 | {'one_hot_max_size': 12}, skip_if_exists=True)                       # the current parameters as trial 0
st.optimize(objective, n_trials=max(0, 40 - len(st.trials)))
print(f'best seed-0 AUC {st.best_value:.4f} (trial 0 = current params: {st.trials[0].value:.4f})')
st.trials_dataframe().sort_values('value', ascending=False).head(8).filter(regex='^(value|params_|user_attrs_rounds)').round(4)

best seed-0 AUC 0.7150 (trial 0 = current params: 0.7146)


,value,params_bagging_temperature,params_border_count,params_depth,params_l2_leaf_reg,params_one_hot_max_size,params_random_strength,user_attrs_rounds
22,0.7150,1.8167,32,4,1.9528,12,0.3387,224
20,0.7148,2.7883,32,3,1.3416,12,0.1634,330
0,0.7146,2.2213,32,4,8.9037,12,0.5161,228
27,0.7146,2.6984,32,4,1.1445,12,0.4183,205
29,0.7145,1.7650,32,3,1.0105,12,0.3939,318
38,0.7145,2.4927,32,3,1.9756,12,0.1264,320
14,0.7145,1.8348,64,3,1.1095,12,0.1116,310
30,0.7144,2.9280,32,4,1.5117,12,0.1762,261


In [5]:
ref15 = C.cv_score(f'cats2_ref15_{len(F1)}', L, F1, 'cb', P1, notes='08 part-1 set, old params') if F1 != F0 else base15
PT = P1 | cb_space(optuna.trial.FixedTrial(st.best_params)) | {'learning_rate': 0.03}
tuned = C.cv_score(f'cats2_tuned15_{len(F1)}', L, F1, 'cb', PT, notes='08 retuned')
d = tuned['fold_aucs'] - ref15['fold_aucs']
res = pd.DataFrame({'Part-1 set, old params': paired(ref15, base15), 'Part-1 set, retuned': paired(tuned, base15)}).T
res['Δ vs old params, seeds 1-2'] = [0, d[5:].mean()]; res['se'] = [0, d[5:].std() / np.sqrt(10)]
display(res.style.format(fmt | {'Δ vs old params, seeds 1-2': '{:+.4f}', 'se': '{:.4f}'}))
retuned = d[5:].mean() > 2 * d[5:].std() / np.sqrt(10)
PF, BEST = (PT, tuned) if retuned else (P1, ref15)
print('Part 2 result:', 'retuned params adopted' if retuned else 'old params kept', f'| 15-fold AUC {BEST["auc_mean"]:.4f}, {BEST["rounds"]} trees')

,auc,Δ auc,se Δ,top_decile,rounds,"Δ vs old params, seeds 1-2",se
"Part-1 set, old params",0.7143,+0.0000,0.0000,30.2%,402.000000,+0.0000,0.0000
"Part-1 set, retuned",0.7139,-0.0004,0.0004,30.0%,366.000000,-0.0007,0.0005


Part 2 result: old params kept | 15-fold AUC 0.7143, 402 trees


## Final: 5-seed submission (if anything changed vs. `07`)

In [6]:
changed = (F1 != F0) or retuned
total = paired(BEST, C.cv_score('fe2_base15', L, json.load(open('params_tuned.json'))['cb']['features'], 'cb',
                                 json.load(open('params_tuned.json'))['cb']['params']))
print(f'vs tuned 41-feature CatBoost (0.7118): Δ {total["Δ auc"]:+.4f} ± {total["se Δ"]:.4f} over 15 folds')
if changed:
    H = FE.add_cats2(FE.add_static(hold))
    preds = np.mean([C.fit_full(L, F1, 'cb', PF | {'random_seed': s}, int(round(BEST['rounds'] * 1.2)))(H) for s in [42, 1, 2, 3, 4]], axis=0)
    out = C.write_submission('catboost_cats2_5', hold, preds)
    json.dump({'features': F1, 'params': PF, 'rounds': BEST['rounds'], 'cv_auc': BEST['auc_mean'], 'cv_top_decile_prec': BEST['top_decile_prec']},
              open('params_cats2.json', 'w'), indent=1, default=str)
    print('wrote', out.name, f'| holdout mean pred {preds.mean():.4f}')
else:
    print('no change vs 07: catboost_cats5 stays the best submission')

vs tuned 41-feature CatBoost (0.7118): Δ +0.0025 ± 0.0004 over 15 folds
no change vs 07: catboost_cats5 stays the best submission


## Conclusions
- **No change: the `07` model stays best** — tuned CatBoost + 4 one-hot copies, 15-fold CV AUC **0.7143** (+0.0025 ± 0.0004 over the tuned 41-feature model). `submissions/catboost_cats5_20261030.csv` remains the submission to use.
- **More one-hot copies:** none pass; 6 of 7 groups hurt, and the numeric originals must stay.
- **Retuning:** the search barely moved (best seed-0 0.7150 vs 0.7146 for the current parameters), and the winner is slightly worse on 15 folds (−0.0007 ± 0.0005 on seeds 1–2). The `04` parameters are still at the optimum for the new set. Best trials again favor depth 3–4 and 32 borders.
- Study `cb_cats2_45` is stored in `optuna.db`.